Regression specifications:
delegitimizing ~ 

In [14]:
import pandas as pd

df = pd.read_csv('full_data.csv')
df.columns

Index(['row_id', 'text-topic', 'text-topic-exp', 'text-entity-name',
       'text-entity-gender', 'text-entity-sentiment',
       'text-entity-sentiment-exp', 'text-generic-frame',
       'text-generic-frame-exp', 'text-issue-frame', 'text-issue-frame-exp',
       'uuid', 'title', 'img-generic-frame', 'img-frame-exp',
       'img-entity-name', 'img-entity-gender', 'img-entity-sentiment',
       'img-entity-sentiment-exp', 'image-url', 'gpt-topic', 'authors',
       'date_publish', 'description', 'language', 'maintext', 'source_domain',
       'url', 'political_leaning', 'month', 'keyword_matches',
       'keyword_match_count', 'word_count', 'label', 'protest_excerpt',
       'excerpt_length', 'gpt_title_frame', 'gpt_excerpt_primary_frame',
       'gpt_excerpt_secondary_frame', 'gpt_demands_present',
       'gpt_protester_quote', 'gpt_authority_quote', 'gpt_thematic_present',
       'frames_error', 'binary_error'],
      dtype='str')

processing

In [15]:
topics_keep = [
    "War",
    "Social Issues",
    "Labor",
    "Legal",
    "Politics",
    "Crime"
]
df["delegit"] = df["gpt_excerpt_primary_frame"].isin(["protester_violence", "disruption", "property_damage"]).astype(int)


sub_df = df[
    df["gpt-topic"].isin(topics_keep)
].copy()


print("Full N:", len(df))
print("Regression N:", len(sub_df))
print("Percent retained:", round(len(sub_df) / len(df) * 100, 1))

sub_df["gpt-topic"].value_counts()

Full N: 9283
Regression N: 7343
Percent retained: 79.1


gpt-topic
Politics         4657
Legal             682
Labor             672
Social Issues     483
Crime             467
War               382
Name: count, dtype: int64

In [12]:
df['delegit'].value_counts()

delegit
0    5315
1    3968
Name: count, dtype: int64

In [19]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm

# -----------------------
# Variables
# -----------------------

binary_cols = [
    #"demands_present",
    "gpt_protester_quote",
    "gpt_authority_quote",
    "gpt_thematic_present"
]

LEAN = 'C(political_leaning, Treatment(reference="center"))'
TOPIC = 'C(Q("gpt-topic"), Treatment(reference="Labor"))'


# -----------------------
# Helper: readable logit output
# -----------------------

def logit_table(model):
    """
    Return odds ratios, confidence intervals, and p-values
    from a fitted statsmodels logistic regression.
    """
    
    conf = model.conf_int()
    
    out = pd.DataFrame({
        "Odds Ratio": np.exp(model.params),
        "CI Low": np.exp(conf[0]),
        "CI High": np.exp(conf[1]),
        "p-value": model.pvalues
    })
    
    return out.round(3)


# Quick checks
print("Full dataset N:", len(df))
print("Six-topic subset N:", len(sub_df))

print("\nPolitical leaning:")
print(df["political_leaning"].value_counts())

print("\nTopics in sub_df:")
print(sub_df["gpt-topic"].value_counts())

Full dataset N: 9283
Six-topic subset N: 7343

Political leaning:
political_leaning
left_lean     4178
right         2441
center        1152
right_lean     921
left           591
Name: count, dtype: int64

Topics in sub_df:
gpt-topic
Politics         4657
Legal             682
Labor             672
Social Issues     483
Crime             467
War               382
Name: count, dtype: int64


In [20]:
# =========================================================
# DELEGITIMIZING FRAME — BINARY LOGISTIC REGRESSIONS
# =========================================================

binary_rhs = " + ".join(binary_cols)


# M1: Political leaning only — FULL DATASET
m1_full = smf.logit(
    f"delegit ~ {LEAN}",
    data=df
).fit()


# M2: Political leaning only — SIX-TOPIC SUBSET
# This is the correct comparison model for M3.
m2_sub = smf.logit(
    f"delegit ~ {LEAN}",
    data=sub_df
).fit()


# M3: Add protest topic — SIX-TOPIC SUBSET
m3_topic = smf.logit(
    f"delegit ~ {LEAN} + {TOPIC}",
    data=sub_df
).fit()


# M4: Add other coverage dimensions — SIX-TOPIC SUBSET
m4_full = smf.logit(
    f"delegit ~ {LEAN} + {TOPIC} + {binary_rhs}",
    data=sub_df
).fit()


# -----------------------
# Readable output
# -----------------------

print("\n=== MODEL 1: Political leaning only — ALL ARTICLES ===")
display(logit_table(m1_full))

print("\n=== MODEL 2: Political leaning only — SIX TOPICS ===")
display(logit_table(m2_sub))

print("\n=== MODEL 3: + Protest topic — SIX TOPICS ===")
display(logit_table(m3_topic))

print("\n=== MODEL 4: + Binary coverage dimensions — SIX TOPICS ===")
display(logit_table(m4_full))

Optimization terminated successfully.
         Current function value: 0.671645
         Iterations 4
Optimization terminated successfully.
         Current function value: 0.677874
         Iterations 4
Optimization terminated successfully.
         Current function value: 0.665144
         Iterations 5
Optimization terminated successfully.
         Current function value: 0.644819
         Iterations 5

=== MODEL 1: Political leaning only — ALL ARTICLES ===


,Odds Ratio,CI Low,CI High,p-value
Intercept,0.625,0.555,0.704,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",0.998,0.814,1.224,0.985
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",0.965,0.844,1.104,0.607
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.945,1.686,2.244,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.160,0.972,1.384,0.100



=== MODEL 2: Political leaning only — SIX TOPICS ===


,Odds Ratio,CI Low,CI High,p-value
Intercept,0.716,0.627,0.818,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",0.921,0.739,1.148,0.463
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",0.920,0.792,1.070,0.279
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.807,1.541,2.120,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.147,0.939,1.401,0.180



=== MODEL 3: + Protest topic — SIX TOPICS ===


,Odds Ratio,CI Low,CI High,p-value
Intercept,0.284,0.228,0.353,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",0.876,0.700,1.095,0.243
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",0.907,0.778,1.058,0.215
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.598,1.357,1.882,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.117,0.911,1.369,0.287
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Crime]",2.341,1.802,3.041,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Legal]",2.051,1.612,2.610,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Politics]",3.121,2.572,3.788,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Social Issues]",3.522,2.715,4.570,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.War]",1.846,1.394,2.443,0.000



=== MODEL 4: + Binary coverage dimensions — SIX TOPICS ===


,Odds Ratio,CI Low,CI High,p-value
Intercept,0.334,0.259,0.432,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",0.909,0.723,1.143,0.414
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",0.920,0.786,1.077,0.300
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.583,1.339,1.872,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.143,0.928,1.407,0.209
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Crime]",2.015,1.538,2.641,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Legal]",1.728,1.348,2.217,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Politics]",2.962,2.428,3.614,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Social Issues]",3.709,2.838,4.846,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.War]",1.747,1.311,2.330,0.000


From M2 (among the six prevalent topics) we see that Right outlets, relative to the Center (reference), give statistically significant increase in odds of delegitimizing frames. From M3, this still holds true even when we control for topics; it is not merely that right outlets cover different topics more.

Interpreting the topics coefficients in M3: we control for political leaning, meaning that social issues having higher rate of delegit is not because social issues is associated with more conservative outlets. Relative to labor: Crime, Legal, Poliitics, War and Social Issues all have stat. significant higher odds of having delegitimizing frame. 

Interpreting binaries: After controlling for outlet lean and topic, articles containing a protester quote have 20% lower odds of being delegit framed compared to articles that do not. Articles with an authority quote are 71% higher odds. Articles with thematic present are 52% lower odds to have delegit frame. 

what if we don't collapse to delegitimizing binary, and keep as dominant frame? 

In [ ]:
# =========================================================
# MULTINOMIAL LOGIT — PRIMARY EXCERPT FRAME
# =========================================================

def prepare_multinomial(df, include_topic=False):
    
    cols = [
        "gpt_excerpt_primary_frame",
        "political_leaning"
    ]
    
    if include_topic:
        cols.append("gpt-topic")
    
    temp = df[cols].dropna().copy()

    # Outcome as categorical codes
    y_cat = pd.Categorical(temp["gpt_excerpt_primary_frame"])
    y = y_cat.codes

    # Political leaning dummies
    # Explicitly set Center as reference by dropping it
    lean_dummies = pd.get_dummies(
        temp["political_leaning"],
        prefix="lean",
        dtype=float
    )

    if "lean_center" in lean_dummies.columns:
        lean_dummies = lean_dummies.drop(columns="lean_center")

    X_parts = [lean_dummies]

    # Topic dummies if requested
    if include_topic:
        topic_dummies = pd.get_dummies(
            temp["gpt-topic"],
            prefix="topic",
            drop_first=True,
            dtype=float
        )
        
        X_parts.append(topic_dummies)

    X = pd.concat(X_parts, axis=1)
    X = sm.add_constant(X)

    return y, X, list(y_cat.categories)


# -----------------------
# Model A: leaning only, full dataset
# -----------------------

y_a, X_a, frame_categories_a = prepare_multinomial(
    df,
    include_topic=False
)

mn_frame_full = sm.MNLogit(y_a, X_a).fit()


# -----------------------
# Model B: leaning + topic, six-topic subset
# -----------------------

y_b, X_b, frame_categories_b = prepare_multinomial(
    sub_df,
    include_topic=True
)

mn_frame_topic = sm.MNLogit(y_b, X_b).fit()


print("\n=== MULTINOMIAL: POLITICAL LEANING ONLY ===")
print("Frame categories:")
print(dict(enumerate(frame_categories_a)))
print(mn_frame_full.summary())


print("\n=== MULTINOMIAL: POLITICAL LEANING + TOPIC ===")
print("Frame categories:")
print(dict(enumerate(frame_categories_b)))
print(mn_frame_topic.summary())

In [23]:
# =========================================================
# BINARY COVERAGE FEATURES
# Each outcome ~ political leaning
#              vs.
# Each outcome ~ political leaning + topic
# =========================================================

binary_models = {}
binary_results = {}

for outcome in binary_cols:
    
    # Same six-topic sample for both models
    temp = sub_df.dropna(
        subset=[outcome, "political_leaning", "gpt-topic"]
    ).copy()
    
    
    # Model A: Political leaning only
    model_lean = smf.logit(
        f"{outcome} ~ {LEAN}",
        data=temp
    ).fit(disp=False)
    
    
    # Model B: Political leaning + topic
    model_topic = smf.logit(
        f"{outcome} ~ {LEAN} + {TOPIC}",
        data=temp
    ).fit(disp=False)
    
    
    binary_models[outcome] = {
        "lean_only": model_lean,
        "lean_plus_topic": model_topic
    }
    
    binary_results[outcome] = {
        "lean_only": logit_table(model_lean),
        "lean_plus_topic": logit_table(model_topic)
    }


# -----------------------
# Display results
# -----------------------

for outcome in binary_cols:
    
    print("\n" + "=" * 70)
    print(outcome.upper())
    print("=" * 70)
    
    print("\nPolitical leaning only:")
    display(binary_results[outcome]["lean_only"])
    
    print("\nPolitical leaning + protest topic:")
    display(binary_results[outcome]["lean_plus_topic"])


GPT_PROTESTER_QUOTE

Political leaning only:


,Odds Ratio,CI Low,CI High,p-value
Intercept,0.753,0.660,0.860,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",1.117,0.899,1.388,0.319
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",1.619,1.395,1.880,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.299,1.108,1.523,0.001
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.169,0.958,1.428,0.124



Political leaning + protest topic:


,Odds Ratio,CI Low,CI High,p-value
Intercept,2.537,2.041,3.154,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",1.180,0.943,1.476,0.148
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",1.700,1.455,1.985,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.484,1.258,1.751,0.000
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.233,1.003,1.515,0.047
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Crime]",0.121,0.092,0.158,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Legal]",0.147,0.115,0.188,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Politics]",0.281,0.231,0.341,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Social Issues]",0.255,0.197,0.332,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.War]",0.295,0.224,0.388,0.000



GPT_AUTHORITY_QUOTE

Political leaning only:


...

,Odds Ratio,CI Low,CI High,p-value
Intercept,2.071,1.667,2.572,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",0.992,0.760,1.296,0.956
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",1.019,0.849,1.224,0.841
"C(political_leaning, Treatment(reference=""center""))[T.right]",1.301,1.061,1.594,0.011
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.119,0.873,1.435,0.376
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Crime]",1.524,1.160,2.003,0.002
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Legal]",1.986,1.541,2.559,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Politics]",2.037,1.699,2.442,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Social Issues]",1.709,1.290,2.264,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.War]",1.841,1.359,2.493,0.000



GPT_THEMATIC_PRESENT

Political leaning only:


,Odds Ratio,CI Low,CI High,p-value
Intercept,1.554,1.359,1.778,0.000
"C(political_leaning, Treatment(reference=""center""))[T.left]",1.240,0.990,1.553,0.062
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",0.947,0.814,1.102,0.480
"C(political_leaning, Treatment(reference=""center""))[T.right]",0.900,0.766,1.057,0.199
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.089,0.888,1.336,0.414



Political leaning + protest topic:


,Odds Ratio,CI Low,CI High,p-value
Intercept,1.348,1.110,1.637,0.003
"C(political_leaning, Treatment(reference=""center""))[T.left]",1.223,0.975,1.534,0.081
"C(political_leaning, Treatment(reference=""center""))[T.left_lean]",0.925,0.794,1.078,0.319
"C(political_leaning, Treatment(reference=""center""))[T.right]",0.828,0.703,0.976,0.024
"C(political_leaning, Treatment(reference=""center""))[T.right_lean]",1.081,0.880,1.327,0.460
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Crime]",0.873,0.688,1.109,0.267
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Legal]",0.857,0.691,1.063,0.160
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Politics]",1.244,1.054,1.468,0.010
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.Social Issues]",2.170,1.681,2.802,0.000
"C(Q(""gpt-topic""), Treatment(reference=""Labor""))[T.War]",1.349,1.041,1.748,0.024


Reinforcing the descriptive table of binaries X topic, we see that topics have stat. significant lower odds of protester quotes than labor (reference), but stat. significantly higher odds of authority quotes relative to labor protest articles.